# 010 Retrieval Complementarity Analysis

This notebook turns the saved dev-only BGE and Qwen3 retrieval predictions into a query-level complementarity report. It does not embed, train, generate answers, or score holdout.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BGE_PREDICTIONS = ROOT / 'data' / 'eval' / 'nonself_retrieval_bge_dev' / 'retrieval_predictions_dev_bge_small_en_v15.jsonl'
QWEN3_PREDICTIONS = ROOT / 'data' / 'eval' / 'nonself_retrieval_qwen3_dev' / 'retrieval_predictions_dev_qwen3_embedding_06b.jsonl'
OUT = ROOT / 'data' / 'eval' / 'nonself_retrieval_complementarity_dev'
SUMMARY_MD = OUT / 'retrieval_complementarity_summary_dev.md'

assert BGE_PREDICTIONS.exists(), f'Missing BGE dev predictions: {BGE_PREDICTIONS}'
assert QWEN3_PREDICTIONS.exists(), f'Missing Qwen3 dev predictions: {QWEN3_PREDICTIONS}'

## Build Artifacts

In [ ]:
RUN_ANALYSIS = True

if RUN_ANALYSIS:
    subprocess.run(
        [
            sys.executable,
            str(ROOT / 'scripts' / 'analyze_retrieval_complementarity.py'),
        ],
        cwd=ROOT,
        check=True,
    )

assert SUMMARY_MD.exists(), f'Missing complementarity summary: {SUMMARY_MD}'

## Summary

In [ ]:
display(Markdown(SUMMARY_MD.read_text(encoding='utf-8')))

## Chart Data

In [ ]:
outcomes = pd.read_csv(OUT / 'retrieval_complementarity_outcome_counts_dev.csv')
upper = pd.read_csv(OUT / 'retrieval_complementarity_upper_bound_dev.csv')
domains = pd.read_csv(OUT / 'retrieval_complementarity_domain_counts_dev.csv')
advantages = pd.read_csv(OUT / 'retrieval_complementarity_rank_advantage_dev.csv')
examples = pd.read_csv(OUT / 'retrieval_complementarity_examples_dev.csv')

display(outcomes)
display(upper)
display(advantages)

## Grade-2 Overlap

In [ ]:
outcome_order = ['both_hit', 'bge_only', 'qwen3_only', 'neither_hit']
outcome_pivot = (
    outcomes.pivot(index='outcome', columns='cutoff', values='queries')
    .reindex(outcome_order)
)
ax = outcome_pivot.plot(kind='bar', figsize=(9, 4), rot=0)
ax.set_title('Grade-2 retrieval overlap by cutoff')
ax.set_xlabel('Outcome')
ax.set_ylabel('Queries')
ax.legend(title='Cutoff')
plt.tight_layout()

## Diagnostic Upper Bound

In [ ]:
ax = upper.plot(
    x='cutoff',
    y=['bge_hit_rate', 'qwen3_hit_rate', 'either_model_hit_rate'],
    marker='o',
    figsize=(8, 4),
)
ax.set_title('Grade-2 hit rate and either-model diagnostic upper bound')
ax.set_xlabel('Cutoff')
ax.set_ylabel('Hit rate')
ax.set_ylim(0, 1)
plt.tight_layout()

## Domain Breakdown

In [ ]:
domain_top5 = domains[
    [
        'expected_primary_domain',
        'eligible_queries',
        'both_hit_at_5',
        'bge_only_at_5',
        'qwen3_only_at_5',
        'neither_hit_at_5',
    ]
]
display(domain_top5)

plot_cols = ['both_hit_at_5', 'bge_only_at_5', 'qwen3_only_at_5', 'neither_hit_at_5']
ax = domain_top5.set_index('expected_primary_domain')[plot_cols].plot(
    kind='barh',
    stacked=True,
    figsize=(9, 4.5),
)
ax.set_title('Grade-2 top-5 complementarity by primary domain')
ax.set_xlabel('Queries')
ax.set_ylabel('Primary domain')
plt.tight_layout()

## Review Examples

In [ ]:
pd.set_option('display.max_colwidth', 120)
display(examples.groupby('example_type', group_keys=False).head(5))

## Working Decision

Do not replace BGE with Qwen3 outright. Treat BGE as the stronger broad top-5 recall candidate and Qwen3 as a useful high-grade ordering/reranking candidate. The next retrieval experiment should test a two-stage or ensemble pattern before the project moves into the classification ladder.